# Real Estate V1 Data Notebook

Use this notebook to inspect the datasets, train a baseline model, and verify the saved model artifacts.

In [17]:
import sys
import os
sys.path.append(os.path.abspath(".."))
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

from src.training.evaluate import evaluate_regression_model

In [18]:
raw_path = Path('../data/raw/california_housing.csv')
cleaned_path = Path('../data/processed/california_housing_cleaned.csv')
feature_path = Path('../data/processed/california_housing_features.csv')
model_path = Path('../models/linear_regression_model.joblib')
feature_columns_path = Path('../models/feature_columns.json')

raw_df = pd.read_csv(raw_path)
cleaned_df = pd.read_csv(cleaned_path)
feature_df = pd.read_csv(feature_path)

In [19]:
print('Raw shape:', raw_df.shape)
print('Cleaned shape:', cleaned_df.shape)
print('Feature shape:', feature_df.shape)

pd.DataFrame(
    {
        'raw_columns': pd.Series(raw_df.columns),
        'cleaned_columns': pd.Series(cleaned_df.columns),
        'feature_columns': pd.Series(feature_df.columns),
    }
)

Raw shape: (20640, 9)
Cleaned shape: (20640, 9)
Feature shape: (20640, 11)


,raw_columns,cleaned_columns,feature_columns
0,MedInc,median_income,median_income
1,HouseAge,house_age,house_age
2,AveRooms,average_rooms,average_rooms
3,AveBedrms,average_bedrooms,average_bedrooms
4,Population,population,population
5,AveOccup,average_occupancy,average_occupancy
6,Latitude,latitude,latitude
7,Longitude,longitude,longitude
8,MedHouseVal,median_house_value,median_house_value
9,NaN,NaN,bedroom_ratio


In [20]:
raw_df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [21]:
cleaned_df.head()

,median_income,house_age,average_rooms,average_bedrooms,population,average_occupancy,latitude,longitude,median_house_value
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [22]:
feature_df[['bedroom_ratio', 'rooms_per_person', 'median_house_value']].head()

,bedroom_ratio,rooms_per_person,median_house_value
0,0.146591,2.732919,4.526
1,0.155797,2.956685,3.585
2,0.129516,2.957661,3.521
3,0.184458,2.283154,3.413
4,0.172096,2.879646,3.422


In [23]:
feature_df.describe()

,median_income,house_age,average_rooms,average_bedrooms,population,average_occupancy,latitude,longitude,median_house_value,bedroom_ratio,rooms_per_person
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558,0.213075,1.976970
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956,0.058023,1.146020
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990,0.100000,0.002547
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000,0.175426,1.522382
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000,0.203181,1.937936
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250,0.239834,2.296090
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010,1.000000,55.222222


## Baseline Model Check

In [24]:
X = feature_df.drop(columns=['median_house_value'])
y = feature_df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
metrics = evaluate_regression_model(y_test, predictions)
metrics

{'rmse': 0.6753472424631182,
 'mae': 0.4861644025414969,
 'r2': 0.6519453808117213}

## Saved Model Artifacts

In [25]:
print('Model exists:', model_path.exists())
print('Feature column file exists:', feature_columns_path.exists())

saved_columns = json.loads(feature_columns_path.read_text())
saved_columns[:5]

Model exists: True
Feature column file exists: True


['median_income',
 'house_age',
 'average_rooms',
 'average_bedrooms',
 'population']

In [26]:
saved_model = joblib.load(model_path)
type(saved_model)

sklearn.linear_model._base.LinearRegression